# Acne lesion detector: training and results

PrecisionSkin / LUMYVUE. One-class YOLOv8s detector that proposes **candidate acne spots** (a box per spot) on face photos. It replaces the acne class of the earlier three-class detector.
This notebook shows **how the final model is trained**, **how its threshold is chosen** and **how it scores** on photos that were never used for training or model selection.

Boxes are candidate lesions, not a diagnosis. The detector is also not acne-specific (it can put boxes on other papular conditions), which is why the classifier and Grad-CAM++ are reported separately.

## 0. How to run
- Run it in WSL with the `.venv_det` environment (ultralytics, torch, pandas, matplotlib).
- `TRAIN = False` (default) uses the finished runs (a few minutes). `TRAIN = True` retrains the three seeds (about 2 hours on an RTX 4060) into a new folder; the finished runs are never overwritten.
- The development test photos are scored once and stored in a lock file. The separate **independent set** (41 photos) is not used in this notebook; its result is in `RESULTS_FOR_PAPER.md`.

In [ ]:
import os, json, time, shutil
from pathlib import Path
TRAIN = False                      # True = retrain the three seeds (slow); False = use the finished runs
QUICK_TEST = False                 # True = 1 epoch of seed 42 only, into a throw-away folder: checks that training works (about 3 minutes)
ALLOW_RESCORE = False
HOME = Path.home() / "PrecisionSkinV1"
DS = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/acne_dataset_v4")
TRAIN_DIR, VAL_DIR, TEST_DIR = DS / "train", DS / "val", DS / "test"
SEEDS = [42, 7, 123]
FINAL_RUNS = {s: HOME / "runs" / "lesion_acne_boxed_only" / "models" / ("seed_%d" % s) for s in SEEDS}
REPORT_RUNS = HOME / "runs" / "report_acne"
LOCK = HOME / "runs" / "report_acne_test_scored.json"
TRAIN_ARGS = dict(imgsz=800, epochs=100, patience=30, batch=8, workers=2, cos_lr=True, close_mosaic=10, fliplr=0.5, flipud=0.0, degrees=10, scale=0.5,
                  hsv_h=0.015, hsv_s=0.5, hsv_v=0.35)
BASE_MODEL = str(HOME / "yolov8s.pt") if (HOME / "yolov8s.pt").exists() else "yolov8s.pt"      # pretrained starting weights
if os.environ.get("NB_QUICK") == "1": TRAIN, QUICK_TEST = True, True
if QUICK_TEST:
    TRAIN = True; SEEDS = [42]; TRAIN_ARGS = dict(TRAIN_ARGS, epochs=1); REPORT_RUNS = HOME / "runs" / "report_acne_QUICKTEST"
    LOCK = HOME / "runs" / "report_acne_QUICKTEST_test.json"
print("mode:", "QUICK TEST (1 epoch, throw-away folder)" if QUICK_TEST else "TRAIN" if TRAIN else "use finished runs")

## 1. Toolbox
Small helper functions: reading boxes, matching predicted boxes to yours, precision / recall / F1, averaging models, plotting. Nothing here trains or changes a model.

In [ ]:
import json, csv, os, sys, time, shutil, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
warnings.filterwarnings("ignore")
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def rd(p):
    p = Path(p); return [tuple(map(float, l.split()[1:5])) for l in p.read_text().splitlines() if len(l.split()) >= 5] if p.exists() else []
def xyxy(b): return b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
def iou(a, b):
    a, b = xyxy(a), xyxy(b); iw, ih = max(0, min(a[2], b[2]) - max(a[0], b[0])), max(0, min(a[3], b[3]) - max(a[1], b[1])); i = iw * ih
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-12)
def side(b): return (b[2] * b[3]) ** 0.5
def images_of(d): return sorted(p for p in (Path(d) / "images").glob("*") if p.suffix.lower() in EXT)

def raw_preds(weights, d, imgsz=800):
    """all boxes down to confidence 0.001 for every photo of folder d (normalised x-centre, y-centre, w, h)"""
    from ultralytics import YOLO
    import torch
    m = YOLO(str(weights)); dev = 0 if torch.cuda.is_available() else "cpu"; out = {}; ims = images_of(d)
    bs = 1 if str(weights).lower().endswith(".onnx") else 8          # exported ONNX files have a fixed batch of 1
    for s in range(0, len(ims), bs):
        ch = ims[s:s + bs]
        for p, r in zip(ch, m.predict([str(x) for x in ch], imgsz=imgsz, conf=0.001, iou=0.6, max_det=300, verbose=False, device=dev)):
            out[p.name] = [(tuple(map(float, b)), float(c)) for b, c in zip(r.boxes.xywhn.cpu().numpy(), r.boxes.conf.cpu().numpy())] if len(r.boxes) else []
    return out

def fuse(lists, thr=0.5):
    """weighted box fusion: boxes of the models that overlap (IoU >= thr) become one box; confidence = sum of confidences / number of models"""
    n = len(lists); allb = sorted([b for l in lists for b in l], key=lambda x: -x[1]); clusters = []
    for b, c in allb:
        for cl in clusters:
            if iou(cl["box"], b) >= thr:
                cl["m"].append((b, c)); w = np.array([x[1] for x in cl["m"]]); cl["box"] = tuple(np.average(np.array([x[0] for x in cl["m"]]), axis=0, weights=w)); break
        else: clusters.append({"box": b, "m": [(b, c)]})
    return [(cl["box"], min(1.0, sum(x[1] for x in cl["m"]) / n)) for cl in clusters]

def _ap(conf, tp, n_gt):
    if n_gt == 0 or len(conf) == 0: return 0.0
    o = np.argsort(-conf); tp = tp[o]; tpc = np.cumsum(tp); fpc = np.cumsum(1 - tp); rec = tpc / n_gt; prec = tpc / (tpc + fpc)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[1], prec, [0]]); mpre = np.flip(np.maximum.accumulate(np.flip(mpre))); x = np.linspace(0, 1, 101)
    return float(np.trapezoid(np.interp(x, mrec, mpre), x)) if hasattr(np, "trapezoid") else float(np.trapz(np.interp(x, mrec, mpre), x))

def match_flags(gt, pr, thr):
    """greedy by confidence: returns (confidences, 1/0 true-positive flags) of every predicted box of one photo"""
    used = set(); cs, fl = [], []
    for b, c in sorted(pr, key=lambda x: -x[1]):
        best, bj = 0, -1
        for j, g in enumerate(gt):
            if j in used: continue
            v = iou(g, b)
            if v > best: best, bj = v, j
        ok = bj >= 0 and best >= thr
        if ok: used.add(bj)
        cs.append(c); fl.append(1.0 if ok else 0.0)
    return cs, fl

def score(preds, d):
    """mAP@0.5, mAP@0.5:0.95 and the raw arrays for precision / recall / F1 at any confidence (ground truth = labels of folder d)"""
    d = Path(d); gts = {k: rd(d / "labels" / (Path(k).stem + ".txt")) for k in preds}; n_gt = sum(len(g) for g in gts.values()); aps = []; conf50 = tp50 = None
    for t in np.arange(0.5, 0.96, 0.05):
        cs, fl = [], []
        for k, pr in preds.items():
            c, f = match_flags(gts[k], pr, t); cs += c; fl += f
        cs, fl = np.array(cs), np.array(fl); aps.append(_ap(cs, fl, n_gt))
        if abs(t - 0.5) < 1e-9: conf50, tp50 = cs, fl
    return dict(AP50=aps[0], AP50_95=float(np.mean(aps)), conf=conf50, tp=tp50, n_gt=n_gt)

def at_conf(s, c):
    m = s["conf"] >= c; tp = s["tp"][m].sum(); p = tp / max(1, m.sum()); r = tp / max(1, s["n_gt"]); return p, r, 2 * p * r / max(1e-9, p + r)

def best_conf(s):
    cs = np.arange(0.05, 0.9, 0.025); return float(max(cs, key=lambda c: at_conf(s, c)[2]))

def rows_at(preds, d, thr):
    """per photo: true positives, false positives, false negatives, zones of your boxes, zones of the model boxes, n your boxes, n model boxes"""
    rows = []
    for k, pr in preds.items():
        gt = rd(Path(d) / "labels" / (Path(k).stem + ".txt")); pc = [b for b, c in pr if c >= thr]; _, fl = match_flags(gt, [(b, 1.0) for b in pc], 0.5); tp = int(sum(fl))
        cell = lambda b: min(2, int(b[1] * 3)) * 3 + min(2, int(b[0] * 3))
        rows.append((tp, len(pc) - tp, len(gt) - tp, {cell(g) for g in gt}, {cell(b) for b in pc}, len(gt), len(pc)))
    return rows

def prf(rows):
    tp, fp, fn = sum(r[0] for r in rows), sum(r[1] for r in rows), sum(r[2] for r in rows); p, r = tp / max(1, tp + fp), tp / max(1, tp + fn); return p, r, 2 * p * r / max(1e-9, p + r)

def zone_f1(rows):
    tp = sum(len(r[3] & r[4]) for r in rows); fp = sum(len(r[4] - r[3]) for r in rows); fn = sum(len(r[3] - r[4]) for r in rows); return 2 * tp / max(1, 2 * tp + fp + fn)

def f1_by_iou(preds, d, thr, ious=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)):
    out = {}
    for t in ious:
        tp = fp = fn = 0
        for k, pr in preds.items():
            gt = rd(Path(d) / "labels" / (Path(k).stem + ".txt")); pc = [(b, 1.0) for b, c in pr if c >= thr]; _, fl = match_flags(gt, pc, t); x = int(sum(fl)); tp += x; fp += len(pc) - x; fn += len(gt) - x
        p, r = tp / max(1, tp + fp), tp / max(1, tp + fn); out[t] = 2 * p * r / max(1e-9, p + r)
    return out

def bootstrap_f1(rows, n=1000, seed=0):
    rng = np.random.default_rng(seed); idx = np.arange(len(rows)); b = [prf([rows[i] for i in rng.choice(idx, len(idx))])[2] for _ in range(n)]; return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

def show_examples(preds, d, thr, names, cols=4, title=""):
    d = Path(d); fig, ax = plt.subplots(-(-len(names) // cols), cols, figsize=(4 * cols, 4 * -(-len(names) // cols))); ax = np.array(ax).reshape(-1)
    for a in ax: a.axis("off")
    for a, n in zip(ax, names):
        p = next(iter((d / "images").glob(Path(n).stem + ".*"))); im = Image.open(p).convert("RGB"); W, H = im.size; dr = ImageDraw.Draw(im); lw = max(2, W // 250)
        for b in rd(d / "labels" / (Path(n).stem + ".txt")): x0, y0, x1, y1 = xyxy(b); dr.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(40, 220, 60), width=lw)
        for b, c in preds[n]:
            if c >= thr: x0, y0, x1, y1 = xyxy(b); dr.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(255, 40, 40), width=lw)
        a.imshow(im); a.set_title(n[:28], fontsize=8)
    fig.suptitle(title + "  (green = your boxes, red = model boxes above the threshold)"); plt.tight_layout(); plt.show()

def curves(run_dirs, labels):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for rd_, lb in zip(run_dirs, labels):
        f = Path(rd_) / "results.csv"
        if not f.exists(): continue
        r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]
        ax[0].plot(r["epoch"], r["train/box_loss"], label=lb + " train"); ax[0].plot(r["epoch"], r["val/box_loss"], "--", label=lb + " val")
        ax[1].plot(r["epoch"], r["metrics/mAP50(B)"], label=lb); ax[2].plot(r["epoch"], r["metrics/mAP50-95(B)"], label=lb)
    for a, t in zip(ax, ("box loss (training = solid, validation = dashed)", "validation mAP@0.5", "validation mAP@0.5:0.95")): a.set_title(t, fontsize=10); a.set_xlabel("epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 2. The data
Acne dataset v4: photos boxed by hand, one box per spot (one class). Training, validation (for choosing models and the threshold) and a development test set (167 photos, hand-boxed, never used for training).
The development test set did guide some earlier decisions (see `ACNE_DETECTOR_VERDICT.md`), so a separate 41-photo independent set was boxed blind and scored once.

In [ ]:
for name, d in (("train", TRAIN_DIR), ("validation", VAL_DIR), ("test", TEST_DIR)):
    ims = images_of(d); nb = [len(rd(d / "labels" / (p.stem + ".txt"))) for p in ims]
    print("%-10s %4d photos | %5d boxes | %.1f boxes per photo | median box %.3f of the photo" % (name, len(ims), sum(nb), np.mean(nb), np.median([side(b) for p in ims for b in rd(d / "labels" / (p.stem + ".txt"))])))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); ax[0].hist([len(rd(TEST_DIR / "labels" / (p.stem + ".txt"))) for p in images_of(TEST_DIR)], bins=20, color="#4c78a8"); ax[0].set_title("boxes per test photo")
ax[1].hist([side(b) * 100 for p in images_of(TEST_DIR) for b in rd(TEST_DIR / "labels" / (p.stem + ".txt"))], bins=30, color="#f58518"); ax[1].set_title("test box size (% of the photo)"); plt.tight_layout(); plt.show()
show_examples({p.name: [] for p in images_of(TEST_DIR)[:8]}, TEST_DIR, 1.0, [p.name for p in images_of(TEST_DIR)[:8]], title="test photos with your boxes")

## 3. Training
YOLOv8s (small), 800 px input, up to 100 epochs, cosine learning-rate decay, mosaic mixing off for the last 10 epochs, flips / small rotations / scale / colour changes as augmentation, early stopping after 30 epochs without improvement.
Three seeds (42, 7, 123); the seed changes the random start and image order. With `TRAIN = False` this cell only prints the settings.

In [ ]:
if TRAIN:
    from ultralytics import YOLO
    import torch, yaml
    REPORT_RUNS.mkdir(parents=True, exist_ok=True)
    yml = REPORT_RUNS / "data.yaml"
    yaml.safe_dump({"path": str(DS), "train": "train/images", "val": "val/images", "names": {0: "acne"}}, open(yml, "w"))
    for s in SEEDS:
        last = REPORT_RUNS / ("seed_%d" % s) / "weights" / "last.pt"
        if last.exists(): YOLO(str(last)).train(resume=True)
        else: YOLO(BASE_MODEL).train(data=str(yml), seed=s, device=0 if torch.cuda.is_available() else "cpu", project=str(REPORT_RUNS), name="seed_%d" % s, exist_ok=True, **TRAIN_ARGS)
    FINAL_RUNS = {s: REPORT_RUNS / ("seed_%d" % s) for s in SEEDS}
else:
    print(json.dumps(TRAIN_ARGS, indent=1))
WEIGHTS = {s: FINAL_RUNS[s] / "weights" / "best.pt" for s in SEEDS}
print({s: p.exists() for s, p in WEIGHTS.items()})

### Training log (from the saved per-epoch records)
The same information as the terminal shows, kept in each run's `results.csv`: one row per epoch. **Every finished epoch** of every seed (a `*` marks an epoch that set a new best mAP@0.5). Scroll the table; it is saved in the notebook, so you can look at it any time without running anything.

In [ ]:
pd.set_option("display.max_rows", None); pd.set_option("display.width", 220)
cols = {"epoch": "epoch", "train/box_loss": "box_loss", "train/cls_loss": "cls_loss", "train/dfl_loss": "dfl_loss", "metrics/precision(B)": "P", "metrics/recall(B)": "R",
        "metrics/mAP50(B)": "mAP50", "metrics/mAP50-95(B)": "mAP50-95", "val/box_loss": "val_box_loss"}
for s in SEEDS:
    f = FINAL_RUNS[s] / "results.csv"
    if not f.exists(): print("seed", s, ": no results.csv yet"); continue
    r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]; t = r[list(cols)].rename(columns=cols)
    t["epoch"] = t["epoch"].astype(int); t["best so far"] = np.where(t["mAP50"] == t["mAP50"].cummax(), "*", "")
    print("=" * 100); print("seed %d  |  %d finished epochs  |  best mAP@0.5 %.3f at epoch %d  |  P, R, mAP are on the validation photos" % (s, len(t), t["mAP50"].max(), int(t.loc[t["mAP50"].idxmax(), "epoch"])))
    display(t.round(3).set_index("epoch"))

## 4. Training curves
Loss should go down; the validation mAP rises, then flattens. The best epoch is kept (`best.pt`).

In [ ]:
curves([FINAL_RUNS[s] for s in SEEDS], ["seed %d" % s for s in SEEDS])

### 4b. Recorded results of the original runs and the figures Ultralytics saved
Everything in this block is read from the finished runs; nothing is re-trained or re-scored. First the numbers written during training (best epoch, validation photos), then the saved figures of each seed.

In [ ]:
from matplotlib import image as mpimg
def show_run_images(fname, title, figw=15, figh=4.8):
    """shows one saved Ultralytics figure of every seed side by side; says so when a file is missing"""
    found = [(s, FINAL_RUNS[s] / fname) for s in SEEDS if (FINAL_RUNS[s] / fname).exists()]
    miss = [s for s in SEEDS if not (FINAL_RUNS[s] / fname).exists()]
    if miss: print("MISSING for seed(s)", miss, "->", fname)
    if not found: return
    fig, ax = plt.subplots(1, len(found), figsize=(figw, figh)); ax = np.atleast_1d(ax)
    for a, (s, p) in zip(ax, found):
        a.imshow(mpimg.imread(str(p))); a.set_title("seed %d" % s, fontsize=10); a.axis("off")
    fig.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()

rec = []
for s in SEEDS:
    f = FINAL_RUNS[s] / "results.csv"
    if not f.exists(): continue
    r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]; i = r["metrics/mAP50(B)"].idxmax()
    rec.append({"seed": s, "epochs finished": len(r), "best epoch": int(r.loc[i, "epoch"]), "precision": r.loc[i, "metrics/precision(B)"], "recall": r.loc[i, "metrics/recall(B)"],
                "mAP@0.5": r.loc[i, "metrics/mAP50(B)"], "mAP@0.5:0.95": r.loc[i, "metrics/mAP50-95(B)"]})
print("Recorded by Ultralytics during training, at the best epoch, on the VALIDATION photos (these are the numbers written in each run's results.csv):")
display(pd.DataFrame(rec).round(3).set_index("seed"))

**Confusion matrices and precision / recall / F1 curves.** The detector has one class, so the confusion matrix has two rows and columns: *lesion* and *background*. The bottom-left cell counts lesions the model missed, the top-right counts boxes it drew where you boxed nothing.

In [ ]:
show_run_images("confusion_matrix.png", "confusion matrix (counts) on the validation photos; a one-class detector has a 2 x 2 matrix: lesion / background")
show_run_images("confusion_matrix_normalized.png", "confusion matrix, normalised per true class")
show_run_images("BoxPR_curve.png", "precision-recall curve (validation photos)")
show_run_images("BoxF1_curve.png", "F1 against confidence (validation photos)")
show_run_images("BoxP_curve.png", "precision against confidence (validation photos)")
show_run_images("BoxR_curve.png", "recall against confidence (validation photos)")

**Training sheets and sample images saved by the runs.**

In [ ]:
show_run_images("results.png", "Ultralytics' own training-curve sheet (losses, precision, recall, mAP per epoch)", figw=18, figh=7)
show_run_images("labels.jpg", "what the training labels look like (box counts, sizes, positions)", figw=18, figh=6)
show_run_images("val_batch0_labels.jpg", "validation photos with YOUR boxes (batch 0)", figw=18, figh=6)
show_run_images("val_batch0_pred.jpg", "the same validation photos with the MODEL's boxes at the end of training (batch 0)", figw=18, figh=6)
show_run_images("train_batch0.jpg", "a training batch as the network sees it (after augmentation)", figw=18, figh=6)

### 4c. Do the training settings documented here match the runs?
The `TRAIN = True` code in section 3 was written to reproduce the runs. This table compares it with `args.yaml`, which Ultralytics wrote at the start of each real run.

In [ ]:
import yaml
keys = ["model", "imgsz", "epochs", "patience", "batch", "workers", "optimizer", "lr0", "cos_lr", "close_mosaic", "fliplr", "flipud", "degrees", "scale", "hsv_h", "hsv_s", "hsv_v", "seed", "pretrained", "amp", "deterministic", "device", "data"]
args = {}
for s in SEEDS:
    f = FINAL_RUNS[s] / "args.yaml"
    if f.exists(): args[s] = yaml.safe_load(open(f))
    else: print("MISSING: args.yaml for seed", s)
if args:
    t = pd.DataFrame({("seed %d" % s): {k: a.get(k, "-") for k in keys} for s, a in args.items()})
    doc = {k: TRAIN_ARGS.get(k, "(not set in TRAIN_ARGS)") for k in keys}
    t.insert(0, "settings in this notebook (TRAIN_ARGS)", pd.Series(doc))
    print("The settings Ultralytics recorded for each finished run (args.yaml) next to the settings this notebook would use if TRAIN = True:")
    display(t)
    diffs = [k for k in TRAIN_ARGS if any(a.get(k) != TRAIN_ARGS[k] for a in args.values())]
    print("settings in TRAIN_ARGS that differ from at least one recorded run:", diffs if diffs else "none")

### 4d. Which original records exist, and which are missing
Checked live on disk. Anything marked MISSING is simply not available; it is not filled in with estimated values.

In [ ]:
def has(p): return "present" if Path(p).exists() else "MISSING"
rows = []
for s in SEEDS:
    d = FINAL_RUNS[s]
    for name, why in (("args.yaml", "exact training settings"), ("results.csv", "one row per epoch"), ("weights/best.pt", "the model that is used"), ("weights/last.pt", "last epoch (resume)"),
                      ("confusion_matrix.png", "confusion matrix"), ("BoxPR_curve.png", "PR curve"), ("train_list.txt", "list of training photos"), ("spec.json", "experiment record"), ("DONE", "run finished marker")):
        rows.append({"seed": s, "artifact": name, "what it is": why, "status": has(d / name)})
t = pd.DataFrame(rows); piv = t.pivot_table(index=["artifact", "what it is"], columns="seed", values="status", aggfunc="first")
display(piv)
logs = FINAL_RUNS[SEEDS[0]].parent.parent / "logs"
if "lesion_acne_boxed_only" in str(FINAL_RUNS[SEEDS[0]]) and logs.exists():
    print("\nExtra records of the acne run (", logs, "):"); print(", ".join(sorted(p.name for p in logs.iterdir())))
    tx = logs / "train_seed_123.txt"
    if tx.exists():
        first = tx.read_text(errors="replace").splitlines()[:2]
        print("\nNOTE: the files train_seed_*.txt in that folder are NOT terminal logs: they contain only a list of training photo paths (%s ...). The terminal output of the real acne training is the executed notebook acne/evidence/Pipeline V5 - Lesion Detector__EXECUTED_by_researcher_2026-10-10.ipynb." % first[0].split("/")[-1])
print("\nNot recorded anywhere I could find (so nothing is invented here):")
print(" - the full terminal log of each seed (per-batch lines) except the one screenshot / executed notebook mentioned above")
print(" - the GPU, wall-clock time and library versions at training time (Ultralytics' args.yaml does not store them; see the evidence notebook for what could be recovered)")
print(" - validation / test scores of the INDIVIDUAL epochs beyond what results.csv contains")

## 5. Validation results per model
Each model scored on the validation photos (used to choose models and thresholds, never the test photos). One row per seed.

In [ ]:
VAL_PRED = {s: raw_preds(WEIGHTS[s], VAL_DIR) for s in SEEDS}
rows = []
for s in SEEDS:
    sc = score(VAL_PRED[s], VAL_DIR); c = best_conf(sc); p, r, f = at_conf(sc, c)
    rows.append(dict(seed=s, mAP50=sc["AP50"], mAP50_95=sc["AP50_95"], threshold=c, precision=p, recall=r, F1=f))
vt = pd.DataFrame(rows); display(vt.round(3)); print("mean mAP@0.5 %.3f (std %.3f)" % (vt.mAP50.mean(), vt.mAP50.std()))

## 6. Averaging the three models and choosing the threshold on validation
Boxes that several models agree on are merged (weighted box fusion). The confidence threshold with the best F1 on the **validation** photos is chosen here, before the test photos are looked at.

In [ ]:
FUSED_VAL = {k: fuse([VAL_PRED[s][k] for s in SEEDS]) for k in VAL_PRED[SEEDS[0]]}
sv = score(FUSED_VAL, VAL_DIR); THR = best_conf(sv); p, r, f = at_conf(sv, THR)
print("3-model average on validation: mAP@0.5 %.3f | mAP@0.5:0.95 %.3f | best F1 %.3f at threshold %.3f (precision %.3f, recall %.3f)" % (sv["AP50"], sv["AP50_95"], f, THR, p, r))
cs = np.arange(0.05, 0.6, 0.025); pr_ = [at_conf(sv, c) for c in cs]
plt.figure(figsize=(6, 3.5)); plt.plot(cs, [x[0] for x in pr_], label="precision"); plt.plot(cs, [x[1] for x in pr_], label="recall"); plt.plot(cs, [x[2] for x in pr_], label="F1", lw=2)
plt.axvline(THR, color="k", ls=":"); plt.xlabel("confidence threshold"); plt.legend(); plt.grid(alpha=.3); plt.title("validation: threshold choice"); plt.show()

## 7. The development test photos (scored once)
167 hand-boxed photos never used for training. The threshold from validation is applied; nothing is tuned here. Note: this set guided some earlier decisions, so the independent set is the more cautious number.

In [ ]:
if LOCK.exists() and not ALLOW_RESCORE:
    R = json.loads(LOCK.read_text()); print("stored result from", R["scored_at"]); TEST_PRED = None
else:
    TEST_PRED = {s: raw_preds(WEIGHTS[s], TEST_DIR) for s in SEEDS}
    FUSED_TEST = {k: fuse([TEST_PRED[s][k] for s in SEEDS]) for k in TEST_PRED[SEEDS[0]]}
    st = score(FUSED_TEST, TEST_DIR); rows_t = rows_at(FUSED_TEST, TEST_DIR, THR); p, r, f = prf(rows_t); lo, hi = bootstrap_f1(rows_t)
    singles = [score(TEST_PRED[s], TEST_DIR)["AP50"] for s in SEEDS]
    R = dict(scored_at=time.strftime("%Y-%m-%d %H:%M"), threshold=THR, mAP50=st["AP50"], mAP50_95=st["AP50_95"], precision=p, recall=r, F1=f, F1_ci=[lo, hi], zone_F1=zone_f1(rows_t),
             photos_with_patch=float(np.mean([x[6] > 0 for x in rows_t])), single_seed_mAP50=singles, n_photos=len(FUSED_TEST), F1_by_iou=f1_by_iou(FUSED_TEST, TEST_DIR, THR),
             count_mae=float(np.mean([abs(x[5] - x[6]) for x in rows_t])), count_within1=float(np.mean([abs(x[5] - x[6]) <= 1 for x in rows_t])),
             count_spearman=float(np.corrcoef(np.argsort(np.argsort([x[5] + 1e-9 * i for i, x in enumerate(rows_t)])), np.argsort(np.argsort([x[6] + 1e-9 * i for i, x in enumerate(rows_t)])))[0, 1]))
    LOCK.parent.mkdir(parents=True, exist_ok=True); LOCK.write_text(json.dumps(R, indent=1)); print("scored and stored in", LOCK)
print("DEVELOPMENT TEST (%d photos), 3-model average at threshold %.3f" % (R["n_photos"], R["threshold"]))
print("  mAP@0.5 %.3f | mAP@0.5:0.95 %.3f | precision %.3f | recall %.3f | F1 %.3f (95%% range %.2f-%.2f)" % (R["mAP50"], R["mAP50_95"], R["precision"], R["recall"], R["F1"], *R["F1_ci"]))
print("  3x3 face-zone F1 %.3f | photos with a box %.0f %% | count: mean error %.2f lesions, Spearman %.2f" % (R["zone_F1"], 100 * R["photos_with_patch"], R["count_mae"], R["count_spearman"]))
print("  single seeds, mAP@0.5:", [round(x, 3) for x in R["single_seed_mAP50"]])
ious = list(map(float, R["F1_by_iou"])); plt.figure(figsize=(5.5, 3.2)); plt.plot(ious, [R["F1_by_iou"][str(i)] if str(i) in R["F1_by_iou"] else R["F1_by_iou"][i] for i in ious], "o-"); plt.xlabel("IoU needed to count a spot as found"); plt.ylabel("F1"); plt.grid(alpha=.3)
plt.title("test: F1 falls as the overlap needed rises (spots are small, about 28 px)"); plt.show()

## 8. Where does it work and where does it fail?

In [ ]:
if TEST_PRED is None:
    TEST_PRED = {s: raw_preds(WEIGHTS[s], TEST_DIR) for s in SEEDS}
FUSED_TEST = {k: fuse([TEST_PRED[s][k] for s in SEEDS]) for k in TEST_PRED[SEEDS[0]]}
sizes = np.array([side(b) for p in images_of(TEST_DIR) for b in rd(TEST_DIR / "labels" / (p.stem + ".txt"))]); q1, q2 = np.percentile(sizes, 33), np.percentile(sizes, 67)
rec = {"small": [0, 0], "medium": [0, 0], "large": [0, 0]}; den = {"sparse (<=6)": [0, 0], "medium (7-14)": [0, 0], "dense (15+)": [0, 0]}
for k, pr in FUSED_TEST.items():
    gt = rd(TEST_DIR / "labels" / (Path(k).stem + ".txt")); pc = [b for b, c in pr if c >= THR]
    for g in gt:
        hit = any(iou(g, b) >= 0.5 for b in pc); z = "small" if side(g) <= q1 else "medium" if side(g) <= q2 else "large"; rec[z][1] += 1; rec[z][0] += hit
        z2 = "sparse (<=6)" if len(gt) <= 6 else "medium (7-14)" if len(gt) <= 14 else "dense (15+)"; den[z2][1] += 1; den[z2][0] += hit
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for a, dct, t in ((ax[0], rec, "recall by spot size"), (ax[1], den, "recall by spots in the photo")):
    a.bar(list(dct), [v[0] / max(1, v[1]) for v in dct.values()], color="#4c78a8"); a.set_ylim(0, 1); a.set_title(t); [a.text(i, v[0] / max(1, v[1]) + .02, "%d/%d" % (v[0], v[1]), ha="center", fontsize=8) for i, v in enumerate(dct.values())]
plt.tight_layout(); plt.show()
tt = []
for thr in (0.15, 0.20, 0.25, 0.275, 0.30, 0.35, 0.40):
    rr = rows_at(FUSED_TEST, TEST_DIR, thr); p, r, f = prf(rr); tt.append(dict(threshold=thr, precision=p, recall=r, F1=f, zone_F1=zone_f1(rr), photos_with_a_box=np.mean([x[6] > 0 for x in rr])))
display(pd.DataFrame(tt).round(3)); print("tiny spots are the main weakness; a higher threshold trades recall for precision")

## 9. Example predictions on test photos

In [ ]:
names = [p.name for p in images_of(TEST_DIR)][8:16]
show_examples(FUSED_TEST, TEST_DIR, THR, names, title="test photos")

## 10. Summary

In [ ]:
ref = pd.DataFrame([("earlier three-class detector (acne class)", "development test", 0.220), ("final single model (seed 123)", "development test", 0.419),
                    ("final single model (seed 123)", "independent set (41 photos, scored once)", 0.358)], columns=["model", "test set", "mAP@0.5"])
ref.loc[len(ref)] = ["THIS NOTEBOOK: 3-model average", "development test", R["mAP50"]]
display(ref.round(3))
print("Final: mAP@0.5 %.3f, F1 %.3f at threshold %.3f on the %d development test photos." % (R["mAP50"], R["F1"], R["threshold"], R["n_photos"]))
print("Usage level (earlier single-model results): photo AUC 0.96, zone F1 0.81-0.82, count error about 4 lesions. Limits: tiny spots are missed, the detector is not acne-specific, only two acne-negative photos in the independent set.")

## 11. Export for the app (optional)
Turns the three models into ONNX files for the C# app: `lesion_detector_acne.onnx` (seed 123 is the main file in the app; here the first seed in `SEEDS` is used) and `..._m2.onnx`, `..._m3.onnx`. Off by default; it overwrites the files in `SkinDevApp/Model`.

In [ ]:
EXPORT = False
if EXPORT:
    from ultralytics import YOLO
    MODEL_DIR = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp/SkinDevApp/Model")
    order = [123, 42, 7]
    for suffix, s in (("", order[0]), ("_m2", order[1]), ("_m3", order[2])):
        o = Path(YOLO(str(WEIGHTS[s])).export(format="onnx", imgsz=800, opset=12, simplify=True, dynamic=False)); shutil.copy2(o, MODEL_DIR / ("lesion_detector_acne%s.onnx" % suffix)); print("exported seed", s)
    print("also set default_conf in lesion_detector_acne.json to %.3f and nms_iou to 0.6" % THR)
else: print("EXPORT = False: nothing written")